# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HasanKhan05/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: Refresh / Content Opportunity Scoring.** This is primarily a **ranking/scoring** task because the decision is not simply whether a page is declining; it is **which pages an editor should review first** when review capacity is limited. The output will be a priority score for each eligible content item, sorted from highest to lowest.

For a content editor deciding what to inspect next, I will score pseudonymized content using observed search, engagement, lifecycle, and content signals. The ranked output supports a human review for refresh, expansion, protection, consolidation, or monitoring. A false positive can waste editorial time, while a false negative can leave a measurable decline unreviewed. The result is decision-support, not proof that an edit will cause recovery.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_file = next(
    candidate / "data/raw/content_refresh_anonymized.csv"
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data/raw/content_refresh_anonymized.csv").exists()
)
raw = pd.read_csv(data_file)
lane_df = raw.loc[raw["impressions_90d"] >= 100].copy()

print(f"Starter rows: {len(raw):,}")
print(f"Reviewable lane rows (impressions_90d >= 100): {len(lane_df):,}")
print("Planned output: one priority score per eligible content item")


Starter rows: 30,000
Reviewable lane rows (impressions_90d >= 100): 22,006
Planned output: one priority score per eligible content item


## 2. Target or proxy

For this starter snapshot, my proxy is **`is_declining_proxy = 1` when the observed `trend_direction` is `down`**. It represents measured decline between the two recent comparison windows. It is useful for learning the workflow, but it is not yet the ideal production target because it describes the current snapshot rather than a later outcome.

The stronger future target would be: *did the content item decline during the next 30 days, using only features measured before that outcome window?* Until that panel target is built, I will call this field a proxy. Because `trend_direction` is derived from `trend_pct`, **neither field may be used as a model feature**.

In [2]:
lane_df["is_declining_proxy"] = (
    lane_df["trend_direction"].str.lower().eq("down").astype(int)
)
proxy_summary = lane_df["is_declining_proxy"].value_counts().rename(
    index={0: "not_down", 1: "down"}
).to_frame("content_items")
proxy_summary["share"] = (proxy_summary["content_items"] / len(lane_df)).round(3)
proxy_summary


,content_items,share
is_declining_proxy,,
down,13152,0.598
not_down,8854,0.402


## 3. Success metric

My primary metric is **Precision@50 on a client-holdout test set**. It answers the operational question: among the 50 content items sent to the editor first, what share have the observed decline proxy? This matches a limited review queue better than overall accuracy.

I will call the result useful when Precision@50 is **at least 5 percentage points above the fixed-rule baseline** on exactly the same held-out clients. The numeric threshold is therefore computed after the baseline rather than chosen in advance. Client holdout matters because content from one pseudonymized client must not appear in both training and evaluation.

In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(labels)[order].mean())

fixed_rule_score = (
    (lane_df["days_since_last_update"] >= 180).astype(int)
    * lane_df["impressions_90d"]
)
baseline_p50 = precision_at_k(fixed_rule_score, lane_df["is_declining_proxy"], 50)
required_p50 = min(1.0, baseline_p50 + 0.05)
print(f"Orientation-only fixed-rule Precision@50 on this full slice: {baseline_p50:.3f}")
print(f"Illustrative useful threshold on this slice: {required_p50:.3f}")
print("The final baseline and threshold will be recomputed on the same client-holdout test set.")


Orientation-only fixed-rule Precision@50 on this full slice: 0.720
Illustrative useful threshold on this slice: 0.770
The final baseline and threshold will be recomputed on the same client-holdout test set.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item at the starter dataset's 90-day snapshot.** The lane slice keeps items with at least 100 impressions so the queue focuses on measurable search exposure. `content_id` identifies the unit but is not a model feature; `client_id` will be used only to create honest client-holdout splits. Rate fields such as `ctr` and `engagement_rate` are percentages on a 0–100 scale.

In [4]:
unit_columns = [
    "content_id", "content_type", "impressions_90d", "avg_position",
    "ctr", "engagement_rate", "content_age_days",
    "days_since_last_update", "trend_direction", "is_declining_proxy"
]
unit_example = lane_df.loc[:, unit_columns].head(8)
print("Unit of analysis: one pseudonymized content item per row")
unit_example


Unit of analysis: one pseudonymized content item per row


,content_id,content_type,impressions_90d,avg_position,ctr,engagement_rate,content_age_days,days_since_last_update,trend_direction,is_declining_proxy
0,content_304f48230142,keyword article,3803,10.6,0.76,5.88,187,20,down,1
1,content_a1fb4e703a9e,keyword article,15320,20.3,0.05,0.00,445,25,down,1
2,content_9aa793d4d895,keyword article,12581,36.5,0.09,0.00,141,20,down,1
3,content_331d6c4de07b,keyword article,11751,6.2,0.49,1.28,463,22,stable,0
4,content_d99b7a2d90ca,keyword article,19140,44.0,0.13,0.00,263,14,down,1
5,content_d4084a4bc775,keyword article,3970,8.5,0.03,0.00,147,20,down,1
7,content_a63219c6e95a,keyword article,1724,21.2,0.06,3.57,445,22,stable,0
8,content_5e6c160719bc,keyword article,32574,46.0,0.09,5.88,90,20,down,1


## 5. Why ML beats a fixed rule here

A fixed rule such as *old page + many impressions* is a useful baseline, but it applies the same thresholds to every content item. The opportunity pattern can depend on interactions among exposure, position, CTR, engagement, age, time since update, and content type. Missingness also differs by content type, so one blanket threshold can confuse *not measured* with *poor performance*.

ML earns a place only if it improves the held-out top-50 queue over the fixed rule while remaining auditable. The output does not automatically order an editor to refresh a page. It narrows the review set; the editor then chooses the appropriate action—refresh, expand, protect, consolidate, or monitor—after examining context.

In [5]:
interaction_check = (
    lane_df.groupby(["content_type", "freshness_tier"], observed=True)
    .agg(
        content_items=("content_id", "size"),
        observed_decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
    )
    .query("content_items >= 30")
    .assign(observed_decline_rate=lambda x: x["observed_decline_rate"].round(3))
    .sort_values("observed_decline_rate", ascending=False)
)
print("Observed proxy rates vary across content-type and freshness combinations:")
interaction_check.head(12)


Observed proxy rates vary across content-type and freshness combinations:


content_items  observed_decline_rate  \
content_type       freshness_tier                                         
keyword article    181+                       35                  0.743   
feedly article     0-30                      222                  0.716   
comparison article 0-30                      366                  0.650   
keyword article    91-180                   7955                  0.623   
feedly article     91-180                    129                  0.612   
keyword article    31-90                     151                  0.596   
                   0-30                    13147                  0.579   

                                   median_impressions  median_ctr  
content_type       freshness_tier                                  
keyword article    181+                         429.0       0.180  
feedly article     0-30                         295.0       0.255  
comparison article 0-30                         229.0       0.000  
keyword article    91-180                      2326.0       0.130  
feedly article     91-180                       469.0       0.210  
keyword article    31-90                        688.0       0.050  
                   0-30                        1579.0       0.160

## 6. Self-check

- [x] Every section above contains both the framing and code that supports it.
- [x] The task type is ranking/scoring, the proxy is named, and Precision@50 is defined before modeling.
- [x] The decision owner is a content editor and the supported action is a prioritized human review.
- [x] The unit of analysis is shown as a real dataframe: one pseudonymized content item per row.
- [x] `trend_direction`, `trend_pct`, `content_id`, and `client_id` will not be model features.
- [x] Claims are limited to observed, measured, directional, and decision-support language.
- [x] No client names, domains, URLs, or private queries are displayed.
- [x] The notebook has been run top to bottom with no errors before submission.